## 1. Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import copy
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.linear_model import Ridge, Lasso
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMRegressor
from scipy.stats import spearmanr

# Load clean transaction data
df_train = pd.read_csv("data/train_clean.csv", parse_dates=["order_date", "pack_date"])
df_valid = pd.read_csv("data/valid_clean.csv", parse_dates=["order_date", "pack_date"])

df_train_targets = pd.read_csv("data/train_targets.csv")
df_valid_targets = pd.read_csv("data/valid_targets.csv")

print("Train transactions:", df_train.shape)
print("Valid transactions:", df_valid.shape)
print("Train targets:     ", df_train_targets.shape)
print("Valid targets:     ", df_valid_targets.shape)

# Reference date: last day of the transaction period
# Used to calculate recency (how many days since last purchase)
reference_date = pd.Timestamp("2017-12-31")

Train transactions: (219128, 27)
Valid transactions: (55183, 27)
Train targets:      (93272, 2)
Valid targets:      (23319, 2)


## 2. Feature Engineering
All features are aggregated at customer level (one row per customer).  
We build three feature blocks and merge them:
- **RFM + behavior features** — recency, frequency, monetary, returns, discounts, delivery, trend
- **Seasonality features** — when during the year the customer buys
- **Product profile features** — what types of products the customer buys

In [2]:
def compute_rfm_features(df):
    """
    Computes RFM and purchase behavior features, aggregated at customer level.
    Input:  transaction-level dataframe (one row per item)
    Output: one row per customer
    """
    df = df.copy()
    # Avoid categorical dtype issues that can cause memory errors
    df["cust_id"] = df["cust_id"].astype(str)
    df["sale_id"]  = df["sale_id"].astype(str)

    # Separate returned and non-returned items
    df_net = df[df["is_returned"] == 0]

    # ── Recency: days since last purchase ──────────────────────────────────
    recency = (
        df.groupby("cust_id")["order_date"]
        .max()
        .apply(lambda x: (reference_date - x).days)
        .rename("recency_days")
    )

    # ── Frequency: unique orders with at least 1 non-returned item ─────────
    frequency = (
        df_net.groupby("cust_id")["sale_id"]
        .nunique()
        .rename("frequency")
    )

    # ── One-time buyer flag ─────────────────────────────────────────────────
    is_one_time_buyer = (frequency == 1).astype(int).rename("is_one_time_buyer")

    # ── Days between orders (only meaningful for repeat buyers) ────────────
    order_dates_per_cust = (
        df.drop_duplicates(subset=["cust_id", "sale_id"])
        .sort_values(["cust_id", "order_date"])
        .groupby("cust_id")["order_date"]
    )
    avg_days_between_orders = (
        order_dates_per_cust
        .apply(lambda x: x.diff().dt.days.mean() if len(x) > 1 else np.nan)
        .rename("avg_days_between_orders")
    )
    std_days_between_orders = (
        order_dates_per_cust
        .apply(lambda x: x.diff().dt.days.std() if len(x) > 1 else np.nan)
        .rename("std_days_between_orders")
    )

    # ── Monetary: total revenue excluding returns ───────────────────────────
    total_revenue = (
        df_net.groupby("cust_id")["sale_revenue"]
        .sum()
        .rename("total_revenue")
    )

    # ── Average ticket: revenue per effective order ─────────────────────────
    avg_ticket = (total_revenue / frequency).rename("avg_ticket")

    # ── Items per order (gross): includes returns ───────────────────────────
    items_per_order_gross = (
        df.groupby(["cust_id", "sale_id"])
        .size()
        .groupby("cust_id").mean()
        .rename("items_per_order_gross")
    )

    # ── Items per order (net): excludes returns ─────────────────────────────
    items_per_order_net = (
        df_net.groupby(["cust_id", "sale_id"])
        .size()
        .groupby("cust_id").mean()
        .rename("items_per_order_net")
    )

    # ── Return behavior ─────────────────────────────────────────────────────
    total_returns = df.groupby("cust_id")["is_returned"].sum().rename("total_returns")
    total_items   = df.groupby("cust_id")["is_returned"].count().rename("total_items")
    return_rate   = (total_returns / total_items).rename("return_rate")

    # ── Delivery time ───────────────────────────────────────────────────────
    df["delivery_days"] = (df["pack_date"] - df["order_date"]).dt.days
    avg_delivery = (
        df.groupby("cust_id")["delivery_days"]
        .mean()
        .rename("avg_delivery_days")
    )

    # ── Discount behavior ───────────────────────────────────────────────────
    df["has_discount_item"] = (df["sale_discount_applied"] < 0).astype(int)
    total_discounted_items = (
        df.groupby("cust_id")["has_discount_item"].sum().rename("total_discounted_items")
    )
    discount_rate = (
        df.groupby("cust_id")["has_discount_item"].mean().rename("discount_rate")
    )

    # ── Trend features: is the customer becoming more active over time? ─────
    df["year"] = df["order_date"].dt.year
    orders_2017  = df[df["year"] == 2017].groupby("cust_id").size()
    orders_total = df.groupby("cust_id").size()
    pct_orders_2017 = (orders_2017 / orders_total).fillna(0).rename("pct_orders_2017")

    revenue_2016 = (
        df[df["year"] == 2016].groupby("cust_id")["sale_revenue"]
        .mean().rename("avg_revenue_2016")
    )
    revenue_2017 = (
        df[df["year"] == 2017].groupby("cust_id")["sale_revenue"]
        .mean().rename("avg_revenue_2017")
    )
    revenue_trend = (revenue_2017 - revenue_2016).rename("revenue_trend")

    rfm = pd.concat([
        recency, frequency, total_revenue, avg_ticket,
        items_per_order_gross, items_per_order_net,
        total_returns, total_items, return_rate,
        total_discounted_items, discount_rate,
        avg_days_between_orders, std_days_between_orders,
        is_one_time_buyer, pct_orders_2017, revenue_trend
    ], axis=1).reset_index()

    return rfm

In [3]:
def compute_seasonality_features(df):
    """
    Captures when during the year the customer tends to buy.
    We focus on January and July (peak sales months).
    most_active_month was discarded due to too many ties.
    """
    df = df.copy()
    df["order_month"] = df["order_date"].dt.month

    buys_in_jan = (
        df.groupby("cust_id")["order_month"]
        .apply(lambda x: int((x == 1).any()))
        .rename("buys_in_jan")
    )
    buys_in_jul = (
        df.groupby("cust_id")["order_month"]
        .apply(lambda x: int((x == 7).any()))
        .rename("buys_in_jul")
    )
    return pd.concat([buys_in_jan, buys_in_jul], axis=1).reset_index()

In [4]:
def compute_product_features(df):
    """
    Captures diversity of purchasing behavior across product categories,
    brands, and customer segments.
    """
    df = df.copy()

    n_genders = df.groupby("cust_id")["prod_type_1"].nunique().rename("n_genders")
    buys_women = (
        df.groupby("cust_id")["prod_type_1"]
        .apply(lambda x: int((x == "women").any())).rename("buys_women")
    )
    buys_men = (
        df.groupby("cust_id")["prod_type_1"]
        .apply(lambda x: int((x == "men").any())).rename("buys_men")
    )
    buys_kids = (
        df.groupby("cust_id")["prod_type_1"]
        .apply(lambda x: int(x.isin(["boys", "girls"]).any())).rename("buys_kids")
    )
    n_product_categories = (
        df.groupby("cust_id")["prod_type_3"].nunique().rename("n_product_categories")
    )
    n_brands = (
        df.groupby("cust_id")["prod_brand"].nunique().rename("n_brands")
    )
    pct_web_only = (
        df.groupby("cust_id")["prod_web_only"].mean().rename("pct_web_only")
    )
    buys_outlet = (
        df.groupby("cust_id")["prod_outlet"]
        .apply(lambda x: int((x > 0).any())).rename("buys_outlet")
    )
    return pd.concat([
        n_genders, buys_women, buys_men, buys_kids,
        n_product_categories, n_brands, pct_web_only, buys_outlet
    ], axis=1).reset_index()

In [5]:
def build_features(df):
    """Combines all feature blocks into one customer-level table."""
    rfm         = compute_rfm_features(df)
    seasonality = compute_seasonality_features(df)
    product     = compute_product_features(df)
    return rfm.merge(seasonality, on="cust_id").merge(product, on="cust_id")

df_train_features = build_features(df_train)
df_valid_features = build_features(df_valid)

print("Train features shape:", df_train_features.shape)
print("Valid features shape:", df_valid_features.shape)

Train features shape: (93272, 27)
Valid features shape: (23319, 27)


In [6]:
# Merge features with targets
df_train_final = df_train_features.merge(df_train_targets, on="cust_id")
df_valid_final = df_valid_features.merge(df_valid_targets, on="cust_id")

print("Train final shape:", df_train_final.shape)
print("Valid final shape:", df_valid_final.shape)

Train final shape: (93272, 28)
Valid final shape: (23319, 28)


## 4. NA Cleaning

In [7]:
target = df_train_final["revenue_2018_2019"]
print("==== Target distribution ====")
print(f"Customers with revenue = 0: {(target==0).sum()} ({100*(target==0).mean():.1f}%)")
print(f"Customers with revenue > 0: {(target>0).sum()} ({100*(target>0).mean():.1f}%)")
print(f"\nMean (all):     {target.mean():.2f}")
print(f"Mean (>0 only): {target[target>0].mean():.2f}")
print(f"Max:            {target.max():.2f}")

print("\n=== NaNs per column ===")
nans = df_train_final.isna().sum()
print(nans[nans > 0])

==== Target distribution ====
Customers with revenue = 0: 59196 (63.5%)
Customers with revenue > 0: 34076 (36.5%)

Mean (all):     70.18
Mean (>0 only): 192.08
Max:            1197.94

=== NaNs per column ===
frequency                      7
total_revenue                  7
avg_ticket                     7
items_per_order_net            7
avg_days_between_orders    61971
std_days_between_orders    78984
is_one_time_buyer              7
revenue_trend              77146
dtype: int64


In [8]:
# ── NaN handling ─────────────────────────────────────────────────────────────
# Customers who returned everything → fill with 0
cols_fill_zero = ["frequency", "total_revenue", "avg_ticket",
                  "items_per_order_net", "is_one_time_buyer"]
df_train_final[cols_fill_zero] = df_train_final[cols_fill_zero].fillna(0)
df_valid_final[cols_fill_zero] = df_valid_final[cols_fill_zero].fillna(0)

# revenue_trend NaN = customer only bought in one year → no trend → fill with 0
df_train_final["revenue_trend"] = df_train_final["revenue_trend"].fillna(0)
df_valid_final["revenue_trend"] = df_valid_final["revenue_trend"].fillna(0)

# One-time buyers → fill days between orders with train median
for col in ["avg_days_between_orders", "std_days_between_orders"]:
    median_val = df_train_final[col].median()
    df_train_final[col] = df_train_final[col].fillna(median_val)
    df_valid_final[col] = df_valid_final[col].fillna(median_val)

print("Remaining NaNs in train:", df_train_final.isna().sum().sum())
print("Remaining NaNs in valid:", df_valid_final.isna().sum().sum())

Remaining NaNs in train: 0
Remaining NaNs in valid: 0


In [9]:
# ── VIP feature ──────────────────────────────────────────────────────────────
# Identified through error analysis: high-value customers are systematically underpredicted
# Thresholds based on train distribution: frequency >= 4 (top quartile), revenue >= 249 (top 10%)
# NOTE: thresholds derived from train only → no data leaking
df_train_final["is_vip"] = (
    (df_train_final["frequency"] >= 4) &
    (df_train_final["total_revenue"] >= 249)
).astype(int)
df_valid_final["is_vip"] = (
    (df_valid_final["frequency"] >= 4) &
    (df_valid_final["total_revenue"] >= 249)
).astype(int)

print("VIP customers in train:", df_train_final["is_vip"].sum())
print("VIP customers in valid:", df_valid_final["is_vip"].sum())

# ── Define feature matrix ─────────────────────────────────────────────────────
feature_cols = [col for col in df_train_final.columns
                if col not in ["cust_id", "revenue_2018_2019"]]

X_train = df_train_final[feature_cols]
y_train = df_train_final["revenue_2018_2019"]
X_valid = df_valid_final[feature_cols]
y_valid = df_valid_final["revenue_2018_2019"]

y_train_binary = (y_train > 0).astype(int)
y_valid_binary = (y_valid > 0).astype(int)

print("\nFeature count:", len(feature_cols))
print("X_train shape:", X_train.shape)

VIP customers in train: 4677
VIP customers in valid: 1202

Feature count: 27
X_train shape: (93272, 27)


## Clustering

In [10]:
# ── K-Means k=5 clustering ───────────────────────────────────────────────────
cluster_cols = ["recency_days", "frequency", "total_revenue",
                "return_rate", "avg_ticket", "pct_orders_2017",
                "n_genders", "avg_days_between_orders", "n_product_categories","buys_kids"]

X_cluster       = df_train_final[cluster_cols].fillna(0)
X_cluster_valid = df_valid_final[cluster_cols].fillna(0)

scaler_km = StandardScaler()
X_cluster_scaled       = scaler_km.fit_transform(X_cluster)
X_cluster_valid_scaled = scaler_km.transform(X_cluster_valid)

# Elbow method showed k=5 optimal (commented out):
#for k in range(2,8): print(f"k={k}, inertia={KMeans(n_clusters=k).fit(X_cluster_scaled).inertia_:.0f}")

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df_train_final["cluster"] = kmeans.fit_predict(X_cluster_scaled)
df_valid_final["cluster"] = kmeans.predict(X_cluster_valid_scaled)

print("=== Cluster profiles ===")
print(df_train_final.groupby("cluster")[["recency_days","frequency","total_revenue","revenue_2018_2019"]].mean().round(2))
print("\nCluster sizes:")
print(df_train_final["cluster"].value_counts().sort_index())

=== Cluster profiles ===
         recency_days  frequency  total_revenue  revenue_2018_2019
cluster                                                           
0              550.83       1.12          88.82              32.44
1              187.47       1.16          90.24              49.98
2              157.07       4.18         387.51             238.21
3              261.82       1.41         104.56              81.02
4              155.92       2.06         170.07             104.93

Cluster sizes:
cluster
0    27033
1    39314
2     8663
3    11549
4     6713
Name: count, dtype: int64


In [11]:
# ── LGBM per cluster — train separate model for each cluster ──────────────────
cluster_models = {}
y_pred_per_cluster = np.zeros(len(df_valid_final))

for cluster_id in range(5):
    train_mask = df_train_final["cluster"] == cluster_id
    valid_mask = df_valid_final["cluster"] == cluster_id

    X_c_train = df_train_final[train_mask][feature_cols]
    y_c_train = np.log1p(df_train_final[train_mask]["revenue_2018_2019"])
    X_c_valid = df_valid_final[valid_mask][feature_cols]
    y_c_valid = df_valid_final[valid_mask]["revenue_2018_2019"]

    model = LGBMRegressor(
        objective="mae", num_leaves=40, min_child_samples=30,
        learning_rate=0.003, n_estimators=5000, subsample=0.8,
        colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
        random_state=42, n_jobs=-1, verbose=-1
    )
    model.fit(X_c_train, y_c_train)
    cluster_models[cluster_id] = model

    pred = np.clip(np.expm1(model.predict(X_c_valid)), 0, None)
    mae  = mean_absolute_error(y_c_valid, pred)

    # Store predictions in correct positions
    valid_idx = df_valid_final[valid_mask].index
    y_pred_per_cluster[valid_idx - df_valid_final.index[0]] = pred

    print(f"Cluster {cluster_id} ({train_mask.sum()} customers) — MAE: {mae:.2f}")

# ── Global results ────────────────────────────────────────────────────────────
mae_cluster      = mean_absolute_error(y_valid, y_pred_per_cluster)
spearman_cluster = spearmanr(y_valid, y_pred_per_cluster).statistic
print(f"\nMAE (LGBM per cluster): {mae_cluster:.2f}")
print(f"Spearman:               {spearman_cluster:.3f}")

Cluster 0 (27033 customers) — MAE: 31.00
Cluster 1 (39314 customers) — MAE: 49.30
Cluster 2 (8663 customers) — MAE: 178.44
Cluster 3 (11549 customers) — MAE: 79.97
Cluster 4 (6713 customers) — MAE: 97.26

MAE (LGBM per cluster): 63.69
Spearman:               0.384


In [12]:
# ── Grid search LGBM per cluster ──────────────────────────────────────────────
# Tested 6 param combinations — best params per cluster:
# Cluster 0: lr=0.002, n=10000, leaves=31 → MAE: 30.98
# Cluster 1: lr=0.002, n=10000, leaves=31 → MAE: 49.29
# Cluster 2: lr=0.005, n=5000,  leaves=31 → MAE: 178.39
# Cluster 3: lr=0.002, n=10000, leaves=31 → MAE: 79.94
# Cluster 4: lr=0.003, n=8000,  leaves=31 → MAE: 96.61

# candidate_params = [
#     {"num_leaves": 31, "min_child_samples": 20, "learning_rate": 0.005, "n_estimators": 5000, "subsample": 0.8, "colsample_bytree": 0.8, "reg_alpha": 0.5, "reg_lambda": 2.0},
#     {"num_leaves": 40, "min_child_samples": 30, "learning_rate": 0.003, "n_estimators": 5000, "subsample": 0.8, "colsample_bytree": 0.8, "reg_alpha": 0.5, "reg_lambda": 2.0},
#     {"num_leaves": 31, "min_child_samples": 20, "learning_rate": 0.003, "n_estimators": 8000, "subsample": 0.8, "colsample_bytree": 0.8, "reg_alpha": 0.5, "reg_lambda": 2.0},
#     {"num_leaves": 31, "min_child_samples": 20, "learning_rate": 0.002, "n_estimators": 10000,"subsample": 0.8, "colsample_bytree": 0.8, "reg_alpha": 0.5, "reg_lambda": 2.0},
#     {"num_leaves": 45, "min_child_samples": 20, "learning_rate": 0.003, "n_estimators": 8000, "subsample": 0.8, "colsample_bytree": 0.8, "reg_alpha": 0.5, "reg_lambda": 2.0},
#     {"num_leaves": 31, "min_child_samples": 15, "learning_rate": 0.003, "n_estimators": 8000, "subsample": 0.8, "colsample_bytree": 0.8, "reg_alpha": 0.5, "reg_lambda": 2.0},
# ]

best_cluster_models = {}
y_pred_grid_cluster = np.zeros(len(df_valid_final))

for cluster_id in range(5):
    train_mask = df_train_final["cluster"] == cluster_id
    valid_mask = df_valid_final["cluster"] == cluster_id

    X_c_train = df_train_final[train_mask][feature_cols]
    y_c_train = np.log1p(df_train_final[train_mask]["revenue_2018_2019"])
    X_c_valid = df_valid_final[valid_mask][feature_cols]
    y_c_valid = df_valid_final[valid_mask]["revenue_2018_2019"]

    if cluster_id == 0:
        m = LGBMRegressor(objective="mae", num_leaves=31, min_child_samples=20,
                          learning_rate=0.002, n_estimators=10000, subsample=0.8,
                          colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
                          random_state=42, n_jobs=-1, verbose=-1)
    elif cluster_id == 1:
        m = LGBMRegressor(objective="mae", num_leaves=31, min_child_samples=20,
                          learning_rate=0.002, n_estimators=10000, subsample=0.8,
                          colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
                          random_state=42, n_jobs=-1, verbose=-1)
    elif cluster_id == 2:
        m = LGBMRegressor(objective="mae", num_leaves=31, min_child_samples=20,
                          learning_rate=0.005, n_estimators=5000, subsample=0.8,
                          colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
                          random_state=42, n_jobs=-1, verbose=-1)
    elif cluster_id == 3:
        m = LGBMRegressor(objective="mae", num_leaves=31, min_child_samples=20,
                          learning_rate=0.002, n_estimators=10000, subsample=0.8,
                          colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
                          random_state=42, n_jobs=-1, verbose=-1)
    elif cluster_id == 4:
        m = LGBMRegressor(objective="mae", num_leaves=31, min_child_samples=20,
                          learning_rate=0.003, n_estimators=8000, subsample=0.8,
                          colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
                          random_state=42, n_jobs=-1, verbose=-1)

    m.fit(X_c_train, y_c_train)
    best_cluster_models[cluster_id] = m

    pred = np.clip(np.expm1(m.predict(X_c_valid)), 0, None)
    mae  = mean_absolute_error(y_c_valid, pred)
    print(f"Cluster {cluster_id} ({train_mask.sum()} customers) — MAE: {mae:.2f}")

    valid_idx = df_valid_final[valid_mask].index
    y_pred_grid_cluster[valid_idx - df_valid_final.index[0]] = pred

mae_grid_cluster      = mean_absolute_error(y_valid, y_pred_grid_cluster)
spearman_grid_cluster = spearmanr(y_valid, y_pred_grid_cluster).statistic
print(f"\nMAE (grid search per cluster): {mae_grid_cluster:.2f}")
print(f"Spearman:                      {spearman_grid_cluster:.3f}")

Cluster 0 (27033 customers) — MAE: 30.98
Cluster 1 (39314 customers) — MAE: 49.29
Cluster 2 (8663 customers) — MAE: 178.39
Cluster 3 (11549 customers) — MAE: 79.94
Cluster 4 (6713 customers) — MAE: 97.27

MAE (grid search per cluster): 63.67
Spearman:                      0.385


In [13]:
# ── Different models per cluster ──────────────────────────────────────────────
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

models_to_try = {
    "LGBM": LGBMRegressor(
        objective="mae", num_leaves=31, min_child_samples=20,
        learning_rate=0.003, n_estimators=8000, subsample=0.8,
        colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
        random_state=42, n_jobs=-1, verbose=-1
    ),
    "XGBoost": XGBRegressor(
        objective="reg:absoluteerror", n_estimators=700, max_depth=6,
        learning_rate=0.01, subsample=0.7, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbosity=0
    ),
    "RF": RandomForestRegressor(
        n_estimators=500, min_samples_leaf=5,
        random_state=42, n_jobs=-1
    ),
}

best_models_per_cluster = {}
y_pred_hetero = np.zeros(len(df_valid_final))

for cluster_id in range(5):
    train_mask = df_train_final["cluster"] == cluster_id
    valid_mask = df_valid_final["cluster"] == cluster_id

    X_c_train = df_train_final[train_mask][feature_cols]
    y_c_train = np.log1p(df_train_final[train_mask]["revenue_2018_2019"])
    X_c_valid = df_valid_final[valid_mask][feature_cols]
    y_c_valid = df_valid_final[valid_mask]["revenue_2018_2019"]

    print(f"\n--- Cluster {cluster_id} ({train_mask.sum()} customers) ---")
    best_mae, best_name, best_model = np.inf, None, None

    for name, model in models_to_try.items():
        m = copy.deepcopy(model)
        m.fit(X_c_train, y_c_train)
        pred = np.clip(np.expm1(m.predict(X_c_valid)), 0, None)
        mae  = mean_absolute_error(y_c_valid, pred)
        print(f"  {name:10s} → MAE: {mae:.2f}")

        if mae < best_mae:
            best_mae   = mae
            best_name  = name
            best_model = m

    best_models_per_cluster[cluster_id] = best_model
    print(f"  → Best: {best_name} ({best_mae:.2f})")

    valid_idx = df_valid_final[valid_mask].index
    y_pred_hetero[valid_idx - df_valid_final.index[0]] = np.clip(
        np.expm1(best_model.predict(X_c_valid)), 0, None
    )

# ── Global results ────────────────────────────────────────────────────────────
mae_hetero      = mean_absolute_error(y_valid, y_pred_hetero)
spearman_hetero = spearmanr(y_valid, y_pred_hetero).statistic
print(f"\nMAE (best model per cluster): {mae_hetero:.2f}")
print(f"Spearman:                     {spearman_hetero:.3f}")


--- Cluster 0 (27033 customers) ---
  LGBM       → MAE: 31.03
  XGBoost    → MAE: 31.02
  RF         → MAE: 32.15
  → Best: XGBoost (31.02)

--- Cluster 1 (39314 customers) ---
  LGBM       → MAE: 49.31
  XGBoost    → MAE: 49.22
  RF         → MAE: 50.65
  → Best: XGBoost (49.22)

--- Cluster 2 (8663 customers) ---
  LGBM       → MAE: 179.30
  XGBoost    → MAE: 177.49
  RF         → MAE: 193.66
  → Best: XGBoost (177.49)

--- Cluster 3 (11549 customers) ---
  LGBM       → MAE: 80.10
  XGBoost    → MAE: 80.08
  RF         → MAE: 82.71
  → Best: XGBoost (80.08)

--- Cluster 4 (6713 customers) ---
  LGBM       → MAE: 97.27
  XGBoost    → MAE: 95.02
  RF         → MAE: 96.08
  → Best: XGBoost (95.02)

MAE (best model per cluster): 63.43
Spearman:                     0.385


In [14]:
# ── Grid search XGBoost per cluster ──────────────────────────────────────────
# Tested 7 param combinations — best params per cluster:
# Cluster 0: n=700,  depth=4, lr=0.010 → MAE: 30.93
# Cluster 1: n=1000, depth=4, lr=0.005 → MAE: 49.08
# Cluster 2: n=1000, depth=4, lr=0.005 → MAE: 176.07
# Cluster 3: n=1000, depth=4, lr=0.005 → MAE: 79.99
# Cluster 4: n=1000, depth=4, lr=0.005 → MAE: 94.04
# Finding: depth=4 won in most clusters → led to v2 grid search focused on depth=4

# candidate_params_xgb = [
#     {"n_estimators": 700,  "max_depth": 6, "learning_rate": 0.01,  "subsample": 0.7, "colsample_bytree": 0.8},
#     {"n_estimators": 1000, "max_depth": 6, "learning_rate": 0.01,  "subsample": 0.7, "colsample_bytree": 0.8},
#     {"n_estimators": 700,  "max_depth": 4, "learning_rate": 0.01,  "subsample": 0.7, "colsample_bytree": 0.8},
#     {"n_estimators": 700,  "max_depth": 6, "learning_rate": 0.005, "subsample": 0.8, "colsample_bytree": 0.8},
#     {"n_estimators": 1000, "max_depth": 6, "learning_rate": 0.005, "subsample": 0.8, "colsample_bytree": 0.8},
#     {"n_estimators": 700,  "max_depth": 8, "learning_rate": 0.01,  "subsample": 0.7, "colsample_bytree": 0.8},
#     {"n_estimators": 1000, "max_depth": 4, "learning_rate": 0.005, "subsample": 0.8, "colsample_bytree": 0.8},
# ]

best_xgb_cluster_models = {}
y_pred_xgb_cluster = np.zeros(len(df_valid_final))

for cluster_id in range(5):
    train_mask = df_train_final["cluster"] == cluster_id
    valid_mask = df_valid_final["cluster"] == cluster_id

    X_c_train = df_train_final[train_mask][feature_cols]
    y_c_train = np.log1p(df_train_final[train_mask]["revenue_2018_2019"])
    X_c_valid = df_valid_final[valid_mask][feature_cols]
    y_c_valid = df_valid_final[valid_mask]["revenue_2018_2019"]

    if cluster_id == 0:
        m = XGBRegressor(objective="reg:absoluteerror", n_estimators=700,  max_depth=4,
                         learning_rate=0.010, subsample=0.7, colsample_bytree=0.8,
                         random_state=42, n_jobs=-1, verbosity=0)
    elif cluster_id == 1:
        m = XGBRegressor(objective="reg:absoluteerror", n_estimators=1000, max_depth=4,
                         learning_rate=0.005, subsample=0.8, colsample_bytree=0.8,
                         random_state=42, n_jobs=-1, verbosity=0)
    elif cluster_id == 2:
        m = XGBRegressor(objective="reg:absoluteerror", n_estimators=1000, max_depth=4,
                         learning_rate=0.005, subsample=0.8, colsample_bytree=0.8,
                         random_state=42, n_jobs=-1, verbosity=0)
    elif cluster_id == 3:
        m = XGBRegressor(objective="reg:absoluteerror", n_estimators=1000, max_depth=4,
                         learning_rate=0.005, subsample=0.8, colsample_bytree=0.8,
                         random_state=42, n_jobs=-1, verbosity=0)
    elif cluster_id == 4:
        m = XGBRegressor(objective="reg:absoluteerror", n_estimators=1000, max_depth=4,
                         learning_rate=0.005, subsample=0.8, colsample_bytree=0.8,
                         random_state=42, n_jobs=-1, verbosity=0)

    m.fit(X_c_train, y_c_train)
    best_xgb_cluster_models[cluster_id] = m

    pred = np.clip(np.expm1(m.predict(X_c_valid)), 0, None)
    mae  = mean_absolute_error(y_c_valid, pred)
    print(f"Cluster {cluster_id} ({train_mask.sum()} customers) — MAE: {mae:.2f}")

    valid_idx = df_valid_final[valid_mask].index
    y_pred_xgb_cluster[valid_idx - df_valid_final.index[0]] = pred

mae_xgb_cluster      = mean_absolute_error(y_valid, y_pred_xgb_cluster)
spearman_xgb_cluster = spearmanr(y_valid, y_pred_xgb_cluster).statistic
print(f"\nMAE (XGBoost grid search per cluster): {mae_xgb_cluster:.2f}")
print(f"Spearman:                              {spearman_xgb_cluster:.3f}")

Cluster 0 (27033 customers) — MAE: 30.93
Cluster 1 (39314 customers) — MAE: 49.08
Cluster 2 (8663 customers) — MAE: 176.07
Cluster 3 (11549 customers) — MAE: 79.99
Cluster 4 (6713 customers) — MAE: 94.04

MAE (XGBoost grid search per cluster): 63.13
Spearman:                              0.396


In [15]:
# ── Grid search XGBoost per cluster v2 — BEST MODEL ──────────────────────────
# Focused on depth=3,4 based on v1 findings — best params per cluster:
# Cluster 0: n=1000, depth=3, lr=0.005 → MAE: 30.89
# Cluster 1: n=1000, depth=3, lr=0.005 → MAE: 49.03
# Cluster 2: n=1000, depth=3, lr=0.005 → MAE: 175.78
# Cluster 3: n=1000, depth=4, lr=0.005 → MAE: 79.90
# Cluster 4: n=1000, depth=4, lr=0.003 → MAE: 93.81

# candidate_params_xgb_v2 = [
#     {"n_estimators": 1000, "max_depth": 4, "learning_rate": 0.005, "subsample": 0.8, "colsample_bytree": 0.8},
#     {"n_estimators": 1500, "max_depth": 4, "learning_rate": 0.005, "subsample": 0.8, "colsample_bytree": 0.8},
#     {"n_estimators": 1000, "max_depth": 4, "learning_rate": 0.003, "subsample": 0.8, "colsample_bytree": 0.8},
#     {"n_estimators": 1500, "max_depth": 4, "learning_rate": 0.003, "subsample": 0.8, "colsample_bytree": 0.8},
#     {"n_estimators": 1000, "max_depth": 4, "learning_rate": 0.005, "subsample": 0.75,"colsample_bytree": 0.75},
#     {"n_estimators": 1000, "max_depth": 4, "learning_rate": 0.005, "subsample": 0.8, "colsample_bytree": 0.8, "reg_alpha": 0.5, "reg_lambda": 2.0},
#     {"n_estimators": 1000, "max_depth": 3, "learning_rate": 0.005, "subsample": 0.8, "colsample_bytree": 0.8},
#     {"n_estimators": 2000, "max_depth": 4, "learning_rate": 0.002, "subsample": 0.8, "colsample_bytree": 0.8},
# ]

best_xgb_v2_models = {}
y_pred_xgb_v2 = np.zeros(len(df_valid_final))

for cluster_id in range(5):
    train_mask = df_train_final["cluster"] == cluster_id
    valid_mask = df_valid_final["cluster"] == cluster_id

    X_c_train = df_train_final[train_mask][feature_cols]
    y_c_train = np.log1p(df_train_final[train_mask]["revenue_2018_2019"])
    X_c_valid = df_valid_final[valid_mask][feature_cols]
    y_c_valid = df_valid_final[valid_mask]["revenue_2018_2019"]

    if cluster_id == 0:
        m = XGBRegressor(objective="reg:absoluteerror", n_estimators=1000, max_depth=3,
                         learning_rate=0.005, subsample=0.8, colsample_bytree=0.8,
                         random_state=42, n_jobs=-1, verbosity=0)
    elif cluster_id == 1:
        m = XGBRegressor(objective="reg:absoluteerror", n_estimators=1000, max_depth=3,
                         learning_rate=0.005, subsample=0.8, colsample_bytree=0.8,
                         random_state=42, n_jobs=-1, verbosity=0)
    elif cluster_id == 2:
        m = XGBRegressor(objective="reg:absoluteerror", n_estimators=1000, max_depth=3,
                         learning_rate=0.005, subsample=0.8, colsample_bytree=0.8,
                         random_state=42, n_jobs=-1, verbosity=0)
    elif cluster_id == 3:
        m = XGBRegressor(objective="reg:absoluteerror", n_estimators=1000, max_depth=4,
                         learning_rate=0.005, subsample=0.8, colsample_bytree=0.8,
                         random_state=42, n_jobs=-1, verbosity=0)
    elif cluster_id == 4:
        m = XGBRegressor(objective="reg:absoluteerror", n_estimators=1000, max_depth=4,
                         learning_rate=0.003, subsample=0.8, colsample_bytree=0.8,
                         random_state=42, n_jobs=-1, verbosity=0)

    m.fit(X_c_train, y_c_train)
    best_xgb_v2_models[cluster_id] = m

    pred = np.clip(np.expm1(m.predict(X_c_valid)), 0, None)
    mae  = mean_absolute_error(y_c_valid, pred)
    print(f"Cluster {cluster_id} ({train_mask.sum()} customers) — MAE: {mae:.2f}")

    valid_idx = df_valid_final[valid_mask].index
    y_pred_xgb_v2[valid_idx - df_valid_final.index[0]] = pred

mae_xgb_v2      = mean_absolute_error(y_valid, y_pred_xgb_v2)
spearman_xgb_v2 = spearmanr(y_valid, y_pred_xgb_v2).statistic
print(f"\nMAE (XGBoost v2 per cluster): {mae_xgb_v2:.2f}")
print(f"Spearman:                     {spearman_xgb_v2:.3f}")

Cluster 0 (27033 customers) — MAE: 30.89
Cluster 1 (39314 customers) — MAE: 49.03
Cluster 2 (8663 customers) — MAE: 175.78
Cluster 3 (11549 customers) — MAE: 79.99
Cluster 4 (6713 customers) — MAE: 93.81

MAE (XGBoost v2 per cluster): 63.05
Spearman:                     0.396


In [16]:
# ── Prepare raw transactions for test customers ───────────────────────────────
df_test_raw = pd.read_csv("data/transactions_2016_2017.csv", low_memory=False,
                           parse_dates=["order_date", "pack_date"])
df_test_customers = pd.read_csv("data/customer_clv_test.csv")

df_test_raw["is_returned"] = (df_test_raw["returned_to_shop_id"].notna()).astype(int)

test_cust_ids = df_test_customers["cust_id"].astype(str).tolist()
df_test_transactions = df_test_raw[df_test_raw["cust_id"].astype(str).isin(test_cust_ids)].copy()

print(f"Test transactions: {df_test_transactions.shape}")
print(f"Test customers found: {df_test_transactions['cust_id'].nunique()}")

# Build features
df_test_features = build_features(df_test_transactions)
df_test_final    = df_test_customers.merge(df_test_features, on="cust_id", how="left")

# NaN handling — same as train
cols_fill_zero = ["frequency", "total_revenue", "avg_ticket",
                  "items_per_order_net", "is_one_time_buyer"]
df_test_final[cols_fill_zero] = df_test_final[cols_fill_zero].fillna(0)
df_test_final["revenue_trend"] = df_test_final["revenue_trend"].fillna(0)
for col in ["avg_days_between_orders", "std_days_between_orders"]:
    df_test_final[col] = df_test_final[col].fillna(df_train_final[col].median())

# VIP feature — same thresholds as train
df_test_final["is_vip"] = (
    (df_test_final["frequency"] >= 4) &
    (df_test_final["total_revenue"] >= 249)
).astype(int)

# ── Assign test customers to clusters ────────────────────────────────────────
X_cluster_test = df_test_final[cluster_cols].fillna(0)
X_cluster_test_scaled = scaler_km.transform(X_cluster_test)
df_test_final["cluster"] = kmeans.predict(X_cluster_test_scaled)

print("\nTest cluster distribution:")
print(df_test_final["cluster"].value_counts().sort_index())

# ── Predict per cluster using best XGBoost models ────────────────────────────
y_pred_test = np.zeros(len(df_test_final))

for cluster_id, model in best_xgb_v2_models.items():
    mask = df_test_final["cluster"] == cluster_id
    X_c_test = df_test_final[mask][feature_cols]
    pred = np.clip(np.expm1(model.predict(X_c_test)), 0, None)
    y_pred_test[mask.values] = pred
    print(f"Cluster {cluster_id}: {mask.sum()} customers predicted")

# ── Save submission ───────────────────────────────────────────────────────────
submission = pd.DataFrame({
    "cust_id":    df_test_final["cust_id"],
    "prediction": y_pred_test
})
submission.to_csv("submission_group21.csv", index=False)
print(f"\nSubmission saved! {submission.shape[0]} customers")
print(f"Mean: {y_pred_test.mean():.2f} | Max: {y_pred_test.max():.2f} | Zeros: {(y_pred_test==0).sum()}")
print(submission.head(10))

Test transactions: (69046, 27)
Test customers found: 29148

Test cluster distribution:
cluster
0     8508
1    12171
2     2736
3     3682
4     2051
Name: count, dtype: int64
Cluster 0: 8508 customers predicted
Cluster 1: 12171 customers predicted
Cluster 2: 2736 customers predicted
Cluster 3: 3682 customers predicted
Cluster 4: 2051 customers predicted

Submission saved! 29148 customers
Mean: 23.56 | Max: 618.98 | Zeros: 18959
            cust_id  prediction
0  2dfoualegmpt6x2h  353.326141
1  d2q2stjpnzld7a4r   11.418094
2  cojscuqlpylhclv2    0.000000
3  vntezlhi2ryvxk6m  125.058754
4  jgy4ytjkdr2b75wf   70.430092
5  z5556xsae7hjyzns    0.000000
6  bsgb4hg5tvtm6c34    0.000000
7  w6helbfkvtpppzjx   41.439884
8  kklab3poqqxo5c2w    0.000000
9  unhzujie5tubmofh    0.016643
